In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/muhammadumer7804/amazon-sales-dataset/amazon.csv


# **Step 1: Import Libraries**

In [2]:
# Step Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('All libraries imported!')

All libraries imported!


# **Step 2: Load Data**

In [3]:
# Step Load the Dataset

path = '/kaggle/input/datasets/muhammadumer7804/amazon-sales-dataset/'

df = pd.read_csv(path + 'amazon.csv')

# check basic info
print('Dataset loaded successfully!')
print('Shape:', df.shape)
print()
print('Column Names:')
print(df.columns.tolist())

Dataset loaded successfully!
Shape: (1465, 16)

Column Names:
['product_id', 'product_name', 'category', 'discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count', 'about_product', 'user_id', 'user_name', 'review_id', 'review_title', 'review_content', 'img_link', 'product_link']


# **Step 3: First Look**

In [4]:
#  First Look at the Data

# see first 5 rows
print('First 5 rows:')
print(df.head())

print()

# check data types
print('Data Types:')
print(df.dtypes)

First 5 rows:
   product_id                                       product_name  \
0  B07JW9H4J1  Wayona Nylon Braided USB to Lightning Fast Cha...   
1  B098NS6PVG  Ambrane Unbreakable 60W / 3A Fast Charging 1.5...   
2  B096MSW6CT  Sounce Fast Phone Charging Cable & Data Sync U...   
3  B08HDJ86NZ  boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...   
4  B08CF3B7N1  Portronics Konnect L 1.2M Fast Charging 3A 8 P...   

                                            category discounted_price  \
0  Computers&Accessories|Accessories&Peripherals|...             ₹399   
1  Computers&Accessories|Accessories&Peripherals|...             ₹199   
2  Computers&Accessories|Accessories&Peripherals|...             ₹199   
3  Computers&Accessories|Accessories&Peripherals|...             ₹329   
4  Computers&Accessories|Accessories&Peripherals|...             ₹154   

  actual_price discount_percentage rating rating_count  \
0       ₹1,099                 64%    4.2       24,269   
1         ₹349        

# **Step 4: Check Missing Values**

In [5]:
#  Check Missing Values

print('Missing Values:')
missing = df.isnull().sum()
print(missing[missing > 0])

print()
print('Total missing:', df.isnull().sum().sum())

Missing Values:
rating_count    2
dtype: int64

Total missing: 2


# **Step 5: Handle Missing Values**

In [6]:
# Handle Missing Values
# only rating_count has 2 missing values
# we drop these rows as they are very few

df.dropna(subset=['rating_count'], inplace=True)

print('Missing values handled!')
print('Rows remaining:', len(df))

Missing values handled!
Rows remaining: 1463


# **Step 6: Check Duplicates**

In [7]:
# Check Duplicate Rows

print('Duplicate rows:', df.duplicated().sum())

# remove duplicates
df.drop_duplicates(inplace=True)

print('After removing duplicates:', len(df))

Duplicate rows: 0
After removing duplicates: 1463


# **Step 7: Fix Data Types**

In [8]:
# Fix Data Types
# price, rating and discount columns have symbols
# we need to remove them and convert to numbers

# remove ₹ sign and commas from price columns
df['discounted_price'] = df['discounted_price'].str.replace('₹', '').str.replace(',', '')
df['actual_price']     = df['actual_price'].str.replace('₹', '').str.replace(',', '')

# remove % sign from discount
df['discount_percentage'] = df['discount_percentage'].str.replace('%', '')

# remove commas from rating count
df['rating_count'] = df['rating_count'].str.replace(',', '')

# now convert to numbers
df['discounted_price']    = pd.to_numeric(df['discounted_price'],    errors='coerce')
df['actual_price']        = pd.to_numeric(df['actual_price'],        errors='coerce')
df['discount_percentage'] = pd.to_numeric(df['discount_percentage'], errors='coerce')
df['rating']              = pd.to_numeric(df['rating'],              errors='coerce')
df['rating_count']        = pd.to_numeric(df['rating_count'],        errors='coerce')

print('Data types fixed!')
print()
print(df[['discounted_price', 'actual_price', 
          'discount_percentage', 'rating']].dtypes)

Data types fixed!

discounted_price       float64
actual_price           float64
discount_percentage      int64
rating                 float64
dtype: object


# **Step 8: Clean Category Column**

In [9]:
#  Clean Category Column
# category has multiple levels separated by |
# we extract only the main category

df['main_category'] = df['category'].str.split('|').str[0]

print('Main categories:')
print(df['main_category'].value_counts())

Main categories:
main_category
Electronics              526
Computers&Accessories    451
Home&Kitchen             448
OfficeProducts            31
MusicalInstruments         2
HomeImprovement            2
Toys&Games                 1
Car&Motorbike              1
Health&PersonalCare        1
Name: count, dtype: int64


# **Step 9: Feature Engineering**

In [10]:
# Step 9 - Feature Engineering
# create new useful columns from existing data

# calculate how much money customer saves
df['amount_saved'] = df['actual_price'] - df['discounted_price']

# calculate savings percentage (verify discount)
df['savings_pct'] = (df['amount_saved'] / df['actual_price'] * 100).round(2)

print('New columns created!')
print()
print(df[['actual_price', 'discounted_price', 
          'amount_saved', 'savings_pct']].head())

New columns created!

   actual_price  discounted_price  amount_saved  savings_pct
0        1099.0             399.0         700.0        63.69
1         349.0             199.0         150.0        42.98
2        1899.0             199.0        1700.0        89.52
3         699.0             329.0         370.0        52.93
4         399.0             154.0         245.0        61.40


# **Step 10: Data Cleaning Summary**

In [11]:
# Step 10 - Data Cleaning Summary

print('Data Cleaning Complete!')
print()
print('Final shape    :', df.shape)
print('Missing values :', df.isnull().sum().sum())
print('Duplicates     :', df.duplicated().sum())
print()
print('Price Range:')
print('Min price      :', df['discounted_price'].min())
print('Max price      :', df['discounted_price'].max())
print('Average price  :', df['discounted_price'].mean().round(2))
print()
print('Rating Range:')
print('Min rating     :', df['rating'].min())
print('Max rating     :', df['rating'].max())
print('Average rating :', df['rating'].mean().round(2))

Data Cleaning Complete!

Final shape    : (1463, 19)
Missing values : 1
Duplicates     : 0

Price Range:
Min price      : 39.0
Max price      : 77990.0
Average price  : 3129.28

Rating Range:
Min rating     : 2.0
Max rating     : 5.0
Average rating : 4.1


# **Fix Remaining Missing Value**

In [12]:
# fix remaining missing value
df.dropna(inplace=True)

print('Missing values:', df.isnull().sum().sum())
print('Final rows:', len(df))

Missing values: 0
Final rows: 1462
